# **Logistics and Last-Mile Delivery Data Quality Testing**
---

###  **Objective**
To identify data quality issues in logistics data using Python and Pandas.

The test cases cover route sequences, proof of delivery, exception
reasons, delivery timestamps, missing events, invalid amounts,
parent shipments, customer privacy, and duplicate tracking IDs.

In [2]:
import pandas as pd
import numpy as np

In [3]:
df0=pd.read_csv('raw_logistics.csv',encoding='latin1')

In [4]:
import copy
df=copy.deepcopy(df0)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Shipment_ID           10000 non-null  object 
 1   Parent_Shipment_ID    374 non-null    object 
 2   Order_Status          10000 non-null  object 
 3   Tracking_ID           10000 non-null  object 
 4   Parcel_ID             10000 non-null  object 
 5   Parcel_Count          10000 non-null  int64  
 6   Carrier               10000 non-null  object 
 7   Service_Level         10000 non-null  object 
 8   Scan_Source           10000 non-null  object 
 9   Event_TS              9825 non-null   object 
 10  Promised_Delivery_TS  9661 non-null   object 
 11  Event_Code            10000 non-null  object 
 12  Status                10000 non-null  object 
 13  Origin_Depot          10000 non-null  object 
 14  Destination_Zone      10000 non-null  object 
 15  Country_Code        

In [5]:
df.head()

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Exception_Reason,Declared_Value,Freight_Cost,Surcharge,COD_Amount,Claim_Amount,Currency,Customer_Name,Customer_Phone,Street_Address
0,SHP-10001,NaN,ACTIVE,trk-9001,PCL-9001,1,FastShip,Exp,DRIVER_APP,2026-07-12 09:00 +05:30,...,NaN,2500.0,120.0,15.0,500.0,0,INR,Ramesh Mohanty,+91-9861012345,"Plot 42, Saheed Nagar, Bhubaneswar"
1,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FAST SHIP,EXPRESS,HUB_SCANNER,2026-07-12T03:30:00Z,...,NaN,2500.0,120.0,15.0,500.0,0,INR,Ramesh Mohanty,9861012345,"Plot 42, Saheed Nagar, Bhubaneswar"
2,SHP-10002,NaN,ACTIVE,TRK-9002,PCL-9002,1,RoadX,Std,DRIVER_APP,2026-07-13 08:00,...,NaN,1800.0,0.0,0.0,0.0,0,INR,Sujata Das,9437054321,"Link Road, Near Nishamani Hall, Cuttack"
3,SHP-10003,NaN,ACTIVE,TRK-9003,PCL-9003,2,Road-X,STANDARD,HUB_SCANNER,2026-07-13T10:00:00Z,...,Damaged,1200.0,210.0,25.0,1200.0,1200,INR,Bikash Jena,8249011223,"Buxi Bazar, Main Road, Cuttack"
4,SHP-10004,SHP-09950,RETURN_RTS,TRK-9002,PCL-9004,1,Fast-Ship,EXPRESS,DRIVER_APP,2026-07-11T04:00:00Z,...,Recipient Unavailable,3200.0,150.0,0.0,0.0,0,INR,Anita Patra,7008122334,"Kalpana Square, Laxmisagar, BBSR"


In [6]:
df.columns

Index(['Shipment_ID', 'Parent_Shipment_ID', 'Order_Status', 'Tracking_ID',
       'Parcel_ID', 'Parcel_Count', 'Carrier', 'Service_Level', 'Scan_Source',
       'Event_TS', 'Promised_Delivery_TS', 'Event_Code', 'Status',
       'Origin_Depot', 'Destination_Zone', 'Country_Code', 'Route_ID',
       'Stop_Seq', 'GPS_Coords', 'Weight', 'Weight_Unit', 'Dimensions_L_W_H',
       'Dim_Unit', 'Attempt_No', 'POD_ID', 'Exception_Reason',
       'Declared_Value', 'Freight_Cost', 'Surcharge', 'COD_Amount',
       'Claim_Amount', 'Currency', 'Customer_Name', 'Customer_Phone',
       'Street_Address'],
      dtype='object')

### 1. Trim and normalize shipment, parcel, route, stop, and tracking IDs; map carrier aliases to canonical identifiers.

In [5]:
df['Shipment_ID']=df['Shipment_ID'].str.strip().str.upper()

df['Tracking_ID']=df['Tracking_ID'].str.strip().str.upper()

df['Route_ID']=df['Route_ID'].str.strip().str.upper()

df['Parcel_ID']=df['Parcel_ID'].str.strip().str.upper()

serv_map={'Next-Day':'NEXTDAY', 'Std':'STANDARD', 'standard':'STANDARD', 'express':'EXPRESS', 'ECONOMY':'ECONOMY', 'Exp':'EXPRESS',
       'EXPRESS':'EXPRESS', 'Eco':'ECONOMY', 'Surface':'SURFACE', 'Regular':'REGULAR', 'STANDARD':'STANDARD'}
df['Service_Level']=df['Service_Level'].replace(serv_map)

df['Event_Code']=df['Event_Code'].replace({'Created':'CREATED', 'Pickup':'PICKED_UP', 'Arrived_At_Hub':'ARRIVED_AT_HUB', 'OFD':'OUT_FOR_DELVERY', 'Deliv_Success':'DELIVERY_SUCCESS',
       'ORDER_CREATED':'ORDER_CREATED', 'picked_up':'PICKED_UP', 'IN_TRANSIT':'IN_TRANSIT', 'Out_For_Delivery':'OUT_FOR_DELVERY',
       'DELIVERED':'DELIVERED', 'Delivered':'DELIVERED', 'PICKED_UP':'PICKED_UP', 'In Transit':'IN_TRANSIT',
       'Out for Delivery':'OUT_FOR_DELIVERY', 'Picked Up':'PICKED_UP', 'delivered':'DELIVERED', 'in_transit':'IN_TRANSIT',
       'out_for_delivery':'OUT_FOR_DELIVERY', 'EXCEPTION':'EXCEPTION', 'created':'CREATED', 'Hold':'HOLD',
       'Label_Generated':'LABEL_GENERATED', 'Delivery_Failed':'DELIVERY_FAILED', 'RTS':"RETURNED_TO_SENDER", 'returned':"RETURNED_TO_SENDER",
       'Exception':'Exception', 'Return_To_Sender':'RETURNED_TO_SENDER', 'Returned':'RETURNED_TO_SENDER'})
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Exception_Reason,Declared_Value,Freight_Cost,Surcharge,COD_Amount,Claim_Amount,Currency,Customer_Name,Customer_Phone,Street_Address
0,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FastShip,EXPRESS,DRIVER_APP,2026-07-12 09:00 +05:30,...,NaN,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,+91-9861012345,"Plot 42, Saheed Nagar, Bhubaneswar"
1,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FAST SHIP,EXPRESS,HUB_SCANNER,2026-07-12T03:30:00Z,...,NaN,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,9861012345,"Plot 42, Saheed Nagar, Bhubaneswar"
2,SHP-10002,NaN,ACTIVE,TRK-9002,PCL-9002,1,RoadX,STANDARD,DRIVER_APP,2026-07-13 08:00,...,NaN,1800.00,0.00,0.0,0.0000,0,INR,Sujata Das,9437054321,"Link Road, Near Nishamani Hall, Cuttack"
3,SHP-10003,NaN,ACTIVE,TRK-9003,PCL-9003,2,Road-X,STANDARD,HUB_SCANNER,2026-07-13T10:00:00Z,...,Damaged,1200.00,210.00,25.0,1200.0000,1200,INR,Bikash Jena,8249011223,"Buxi Bazar, Main Road, Cuttack"
4,SHP-10004,SHP-09950,RETURN_RTS,TRK-9002,PCL-9004,1,Fast-Ship,EXPRESS,DRIVER_APP,2026-07-11T04:00:00Z,...,Recipient Unavailable,3200.00,150.00,0.0,0.0000,0,INR,Anita Patra,7008122334,"Kalpana Square, Laxmisagar, BBSR"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,Odisha-Exp,NEXTDAY,API_WEBHOOK,2026-07-10 10:54 +05:30,...,NaN,8531.66,209.81,25.0,450.0000,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar"
9996,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,FastShip,NEXTDAY,HUB_SCANNER,2026-07-10T12:51:00Z,...,NaN,8531.66,209.81,25.0,450.0000,0,NaN,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar"
9997,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,Odisha-Exp,NEXTDAY,DRIVER_APP,2026-07-11 00:04 +05:30,...,NaN,8531.66,0.00,25.0,0.4500,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar"
9998,TEST-11916,NaN,TEST_ORDER,TRK-11916,PCL-11916,0,OdishaExpress,EXPRESS,DRIVER_APP,2026-07-23 10:43 +05:30,...,NaN,4732.47,0.00,50.0,2499.0000,0,INR,NaN,9069150103,"Fakir Mohan Golai, Station Road, Balasore"


In [6]:
carrier_map={'FastShip':'FASTSHIP', 'FAST SHIP':'FASTSHIP', 'RoadX':'ROADX', 'Road-X':'ROADX', 'Fast-Ship':'FASTSHIP',
       'BLUE JET':'BLUEJET', 'BlueJet':'BLUEJET', 'ROAD_X':'ROADX', 'ODISHA EXPRESS':'ODISHAEXPRESS', 'Blue-Jet':'BLUEJET',
       'Odisha-Exp':'ODISHAEXPRESS', 'OdishaExpress':'ODISHAEXPRESS', 'roadx_logistics':'ROADX', 'fastship_exp':'FASTSHIP'}
df['Carrier'] = df['Carrier'].map(carrier_map).fillna(df['Carrier'])

In [9]:
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Exception_Reason,Declared_Value,Freight_Cost,Surcharge,COD_Amount,Claim_Amount,Currency,Customer_Name,Customer_Phone,Street_Address
0,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-12 09:00 +05:30,...,NaN,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,+91-9861012345,"Plot 42, Saheed Nagar, Bhubaneswar"
1,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FASTSHIP,EXPRESS,HUB_SCANNER,2026-07-12T03:30:00Z,...,NaN,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,9861012345,"Plot 42, Saheed Nagar, Bhubaneswar"
2,SHP-10002,NaN,ACTIVE,TRK-9002,PCL-9002,1,ROADX,STANDARD,DRIVER_APP,2026-07-13 08:00,...,NaN,1800.00,0.00,0.0,0.0000,0,INR,Sujata Das,9437054321,"Link Road, Near Nishamani Hall, Cuttack"
3,SHP-10003,NaN,ACTIVE,TRK-9003,PCL-9003,2,ROADX,STANDARD,HUB_SCANNER,2026-07-13T10:00:00Z,...,Damaged,1200.00,210.00,25.0,1200.0000,1200,INR,Bikash Jena,8249011223,"Buxi Bazar, Main Road, Cuttack"
4,SHP-10004,SHP-09950,RETURN_RTS,TRK-9002,PCL-9004,1,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-11T04:00:00Z,...,Recipient Unavailable,3200.00,150.00,0.0,0.0000,0,INR,Anita Patra,7008122334,"Kalpana Square, Laxmisagar, BBSR"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,ODISHAEXPRESS,NEXTDAY,API_WEBHOOK,2026-07-10 10:54 +05:30,...,NaN,8531.66,209.81,25.0,450.0000,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar"
9996,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,FASTSHIP,NEXTDAY,HUB_SCANNER,2026-07-10T12:51:00Z,...,NaN,8531.66,209.81,25.0,450.0000,0,NaN,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar"
9997,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,ODISHAEXPRESS,NEXTDAY,DRIVER_APP,2026-07-11 00:04 +05:30,...,NaN,8531.66,0.00,25.0,0.4500,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar"
9998,TEST-11916,NaN,TEST_ORDER,TRK-11916,PCL-11916,0,ODISHAEXPRESS,EXPRESS,DRIVER_APP,2026-07-23 10:43 +05:30,...,NaN,4732.47,0.00,50.0,2499.0000,0,INR,NaN,9069150103,"Fakir Mohan Golai, Station Road, Balasore"


### 2. Parse event timestamps and normalize timezone offsets to UTC while retaining source timezone when useful.

In Pandas, errors='coerce' means if a value cannot be converted into the required data type, Pandas replaces it with a missing value (NaN or NaT) instead of raising an error.
* NaT stands for Not a Time

In Pandas, utc=True in pd.to_datetime() means convert or interpret datetime values as UTC (Coordinated Universal Time) and return timezone-aware datetime values.

In [7]:
df['Event_TS_UTC'] = pd.to_datetime(df['Event_TS'], errors='coerce', utc=True)
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Declared_Value,Freight_Cost,Surcharge,COD_Amount,Claim_Amount,Currency,Customer_Name,Customer_Phone,Street_Address,Event_TS_UTC
0,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-12 09:00 +05:30,...,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,+91-9861012345,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-12 03:30:00+00:00
1,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FASTSHIP,EXPRESS,HUB_SCANNER,2026-07-12T03:30:00Z,...,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,9861012345,"Plot 42, Saheed Nagar, Bhubaneswar",NaT
2,SHP-10002,NaN,ACTIVE,TRK-9002,PCL-9002,1,ROADX,STANDARD,DRIVER_APP,2026-07-13 08:00,...,1800.00,0.00,0.0,0.0000,0,INR,Sujata Das,9437054321,"Link Road, Near Nishamani Hall, Cuttack",NaT
3,SHP-10003,NaN,ACTIVE,TRK-9003,PCL-9003,2,ROADX,STANDARD,HUB_SCANNER,2026-07-13T10:00:00Z,...,1200.00,210.00,25.0,1200.0000,1200,INR,Bikash Jena,8249011223,"Buxi Bazar, Main Road, Cuttack",NaT
4,SHP-10004,SHP-09950,RETURN_RTS,TRK-9002,PCL-9004,1,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-11T04:00:00Z,...,3200.00,150.00,0.0,0.0000,0,INR,Anita Patra,7008122334,"Kalpana Square, Laxmisagar, BBSR",NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,ODISHAEXPRESS,NEXTDAY,API_WEBHOOK,2026-07-10 10:54 +05:30,...,8531.66,209.81,25.0,450.0000,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 05:24:00+00:00
9996,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,FASTSHIP,NEXTDAY,HUB_SCANNER,2026-07-10T12:51:00Z,...,8531.66,209.81,25.0,450.0000,0,NaN,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",NaT
9997,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,ODISHAEXPRESS,NEXTDAY,DRIVER_APP,2026-07-11 00:04 +05:30,...,8531.66,0.00,25.0,0.4500,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 18:34:00+00:00
9998,TEST-11916,NaN,TEST_ORDER,TRK-11916,PCL-11916,0,ODISHAEXPRESS,EXPRESS,DRIVER_APP,2026-07-23 10:43 +05:30,...,4732.47,0.00,50.0,2499.0000,0,INR,NaN,9069150103,"Fakir Mohan Golai, Station Road, Balasore",2026-07-23 05:13:00+00:00


In [8]:
# Check whether the original timestamp contains a timezone
timestamp_text = df['Event_TS'].astype('string').str.strip()

has_timezone = timestamp_text.str.contains(
    r'(?:Z|[+-]\d{2}:?\d{2})$',
    case=False,
    na=False
)

# Flag timestamps without timezone information
df['Timezone_Status'] = np.where(
    has_timezone,
    'VALID_TIMEZONE',
    'REVIEW_MISSING_TIMEZONE'
)

# Convert only timestamps with an explicit timezone
df['Event_TS_Validated'] = pd.to_datetime(
    timestamp_text.where(has_timezone),
    errors='coerce',
    utc=True
)

# Display records requiring review
timezone_issues = df[
    df['Timezone_Status'] == 'REVIEW_MISSING_TIMEZONE'
]

print("Records missing timezone:", len(timezone_issues))

display(
    timezone_issues[
        ['Event_TS', 'Timezone_Status', 'Event_TS_Validated']
    ].head(10)
)

Records missing timezone: 989


,Event_TS,Timezone_Status,Event_TS_Validated
2,2026-07-13 08:00,REVIEW_MISSING_TIMEZONE,NaT
7,NaN,REVIEW_MISSING_TIMEZONE,NaT
8,2026-07-23 19:35,REVIEW_MISSING_TIMEZONE,NaT
9,2026-07-24 03:07,REVIEW_MISSING_TIMEZONE,NaT
11,2026-07-09 17:33,REVIEW_MISSING_TIMEZONE,NaT
23,2026-07-24 04:40,REVIEW_MISSING_TIMEZONE,NaT
26,2026-07-25 06:13,REVIEW_MISSING_TIMEZONE,NaT
37,2026-07-10 02:21,REVIEW_MISSING_TIMEZONE,NaT
40,2026-07-11 16:35,REVIEW_MISSING_TIMEZONE,NaT
47,2026-07-30 11:18,REVIEW_MISSING_TIMEZONE,NaT


### 3. Standardize carrier, service level, warehouse, depot, and delivery-zone names using reference tables.

In [11]:
df['Origin_Depot'].unique()

array(['WH-N', 'WH-NORTH', 'WH-S', 'WH-SOUTH', 'WH-EAST', 'WH-WEST',
       'South-Hub', 'East-Hub', 'West-Hub', 'WH-W', 'North-Hub', 'WH-E'],
      dtype=object)

In [12]:
depot_map = {'WH-N':'NORTH', 'WH-NORTH':'NORTH', 'WH-S':'SOUTH', 'WH-SOUTH':'SOUTH', 'WH-EAST':'EAST', 
             'WH-WEST':'WEST','South-Hub':'SOUTH', 'East-Hub':'EAST', 'West-Hub':'WEST', 'WH-W':'WEST', 'North-Hub':'NORTH', 'WH-E':'EAST'}
df['Origin_Depot'] = df['Origin_Depot'].map(depot_map).fillna(df['Origin_Depot'])
df['Origin_Depot'].unique()

array(['NORTH', 'SOUTH', 'EAST', 'WEST'], dtype=object)

### 4. Normalize shipment statuses (created, picked up, in transit, out for delivery, delivered, returned, exception).

In [13]:
df['Status']=df['Status'].str.strip().str.upper().str.replace(' ','_')
df['Status'].unique()

array(['PICKED_UP', 'DELIVERED', 'IN_TRANSIT', 'CREATED',
       'OUT_FOR_DELIVERY', 'HOLD', 'RTS', 'LABEL_GENERATED', 'PICKUP',
       'ORDER_CREATED', 'DELIV_SUCCESS', 'EXCEPTION', 'ARRIVED_AT_HUB',
       'OFD', 'DELIVERY_FAILED', 'RETURNED', 'RETURN_TO_SENDER'],
      dtype=object)

In [14]:
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Declared_Value,Freight_Cost,Surcharge,COD_Amount,Claim_Amount,Currency,Customer_Name,Customer_Phone,Street_Address,Event_TS_UTC
0,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-12 09:00 +05:30,...,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,+91-9861012345,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-12 03:30:00+00:00
1,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FASTSHIP,EXPRESS,HUB_SCANNER,2026-07-12T03:30:00Z,...,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,9861012345,"Plot 42, Saheed Nagar, Bhubaneswar",NaT
2,SHP-10002,NaN,ACTIVE,TRK-9002,PCL-9002,1,ROADX,STANDARD,DRIVER_APP,2026-07-13 08:00,...,1800.00,0.00,0.0,0.0000,0,INR,Sujata Das,9437054321,"Link Road, Near Nishamani Hall, Cuttack",NaT
3,SHP-10003,NaN,ACTIVE,TRK-9003,PCL-9003,2,ROADX,STANDARD,HUB_SCANNER,2026-07-13T10:00:00Z,...,1200.00,210.00,25.0,1200.0000,1200,INR,Bikash Jena,8249011223,"Buxi Bazar, Main Road, Cuttack",NaT
4,SHP-10004,SHP-09950,RETURN_RTS,TRK-9002,PCL-9004,1,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-11T04:00:00Z,...,3200.00,150.00,0.0,0.0000,0,INR,Anita Patra,7008122334,"Kalpana Square, Laxmisagar, BBSR",NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,ODISHAEXPRESS,NEXTDAY,API_WEBHOOK,2026-07-10 10:54 +05:30,...,8531.66,209.81,25.0,450.0000,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 05:24:00+00:00
9996,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,FASTSHIP,NEXTDAY,HUB_SCANNER,2026-07-10T12:51:00Z,...,8531.66,209.81,25.0,450.0000,0,NaN,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",NaT
9997,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,ODISHAEXPRESS,NEXTDAY,DRIVER_APP,2026-07-11 00:04 +05:30,...,8531.66,0.00,25.0,0.4500,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 18:34:00+00:00
9998,TEST-11916,NaN,TEST_ORDER,TRK-11916,PCL-11916,0,ODISHAEXPRESS,EXPRESS,DRIVER_APP,2026-07-23 10:43 +05:30,...,4732.47,0.00,50.0,2499.0000,0,INR,NaN,9069150103,"Fakir Mohan Golai, Station Road, Balasore",2026-07-23 05:13:00+00:00


### 5. Deduplicate repeated scan events using tracking ID, event code, timestamp, location, and source priority.

In [ ]:
# Standardize text
df['Tracking_ID'] = df['Tracking_ID'].str.strip().str.upper()

df['Event_Code'] = (
    df['Event_Code'].astype('string').str.strip().str.upper()
)

df['Origin_Depot'] = (
    df['Origin_Depot'].astype('string').str.strip().str.upper()
)

# Normalize timestamps to UTC
df['Event_TS_UTC'] = pd.to_datetime(
    df['Event_TS'], errors='coerce', utc=True
)

# Convert weights to kilograms
df['Weight_kg'] = np.where(
    df['Weight_Unit'].str.lower().eq('g'),
    df['Weight'] / 1000,
    np.where(
        df['Weight_Unit'].str.lower().eq('kg'),
        df['Weight'],
        np.nan
    )
)

# Identify equivalent scan events
duplicate_key = [
    'Tracking_ID',
    'Event_Code',
    'Event_TS_UTC',
    'Origin_Depot'
]

# Remove equivalent scans
df = df.drop_duplicates(
    subset=duplicate_key,
    keep='first'
).copy()

print("Remaining records:", len(df))

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Declared_Value,Freight_Cost,Surcharge,COD_Amount,Claim_Amount,Currency,Customer_Name,Customer_Phone,Street_Address,Event_TS_UTC
0,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-12 09:00 +05:30,...,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,+91-9861012345,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-12 03:30:00+00:00
1,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1,FASTSHIP,EXPRESS,HUB_SCANNER,2026-07-12T03:30:00Z,...,2500.00,120.00,15.0,500.0000,0,INR,Ramesh Mohanty,9861012345,"Plot 42, Saheed Nagar, Bhubaneswar",NaT
2,SHP-10002,NaN,ACTIVE,TRK-9002,PCL-9002,1,ROADX,STANDARD,DRIVER_APP,2026-07-13 08:00,...,1800.00,0.00,0.0,0.0000,0,INR,Sujata Das,9437054321,"Link Road, Near Nishamani Hall, Cuttack",NaT
3,SHP-10003,NaN,ACTIVE,TRK-9003,PCL-9003,2,ROADX,STANDARD,HUB_SCANNER,2026-07-13T10:00:00Z,...,1200.00,210.00,25.0,1200.0000,1200,INR,Bikash Jena,8249011223,"Buxi Bazar, Main Road, Cuttack",NaT
4,SHP-10004,SHP-09950,RETURN_RTS,TRK-9002,PCL-9004,1,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-11T04:00:00Z,...,3200.00,150.00,0.0,0.0000,0,INR,Anita Patra,7008122334,"Kalpana Square, Laxmisagar, BBSR",NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,ODISHAEXPRESS,NEXTDAY,API_WEBHOOK,2026-07-10 10:54 +05:30,...,8531.66,209.81,25.0,450.0000,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 05:24:00+00:00
9996,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,FASTSHIP,NEXTDAY,HUB_SCANNER,2026-07-10T12:51:00Z,...,8531.66,209.81,25.0,450.0000,0,NaN,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",NaT
9997,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1,ODISHAEXPRESS,NEXTDAY,DRIVER_APP,2026-07-11 00:04 +05:30,...,8531.66,0.00,25.0,0.4500,0,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 18:34:00+00:00
9998,TEST-11916,NaN,TEST_ORDER,TRK-11916,PCL-11916,0,ODISHAEXPRESS,EXPRESS,DRIVER_APP,2026-07-23 10:43 +05:30,...,4732.47,0.00,50.0,2499.0000,0,INR,NaN,9069150103,"Fakir Mohan Golai, Station Road, Balasore",2026-07-23 05:13:00+00:00


### 6. Normalize weight and dimension units (g/kg, cm/m, lb/in) with explicit conversion rules.

In [16]:
# 1. Convert Weights (g and lb) into kg
df['Weight_kg'] = np.where(df['Weight_Unit'] == 'g', df['Weight'] / 1000.0, 
                  np.where(df['Weight_Unit'] == 'lb', df['Weight'] * 0.453592, df['Weight'])).round(2)

# 2. Split the '30x20x10' string into three temporary numeric columns
dims = df['Dimensions_L_W_H'].astype(str).str.split('x', expand=True).apply(pd.to_numeric, errors='coerce')

# 3. Convert Dimensions (m and in) into cm based on the Dim_Unit column
df['Length_cm'] = np.where(df['Dim_Unit'] == 'm', dims[0] * 100.0, 
                  np.where(df['Dim_Unit'] == 'in', dims[0] * 2.54, dims[0]))

df['Width_cm'] = np.where(df['Dim_Unit'] == 'm', dims[1] * 100.0, 
                 np.where(df['Dim_Unit'] == 'in', dims[1] * 2.54, dims[1]))

df['Height_cm'] = np.where(df['Dim_Unit'] == 'm', dims[2] * 100.0, 
                  np.where(df['Dim_Unit'] == 'in', dims[2] * 2.54, dims[2]))

### 7. Validate package weight, dimensions, declared value, and parcel count against allowable ranges.

In [17]:
# 1. Nullify weights that are 0, negative, or too heavy (over 500 kg)
df.loc[(df['Weight_kg'] <= 0) | (df['Weight_kg'] > 500), 'Weight_kg'] = np.nan

# 2. Nullify parcel counts that are 0, negative, or unrealistically high (over 50)
df.loc[(df['Parcel_Count'] <= 0) | (df['Parcel_Count'] > 50), 'Parcel_Count'] = np.nan

# 3. Nullify declared monetary values that are negative
df.loc[df['Declared_Value'] < 0, 'Declared_Value'] = np.nan

# 4. Simply check if Dimensions have exactly two 'x' characters (e.g., '30x20x10')
df['Dimensions_Valid'] = np.where(df['Dimensions_L_W_H'].astype(str).str.count('x') == 2, 1, 0)

In [18]:
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Currency,Customer_Name,Customer_Phone,Street_Address,Event_TS_UTC,Weight_kg,Length_cm,Width_cm,Height_cm,Dimensions_Valid
0,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1.0,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-12 09:00 +05:30,...,INR,Ramesh Mohanty,+91-9861012345,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-12 03:30:00+00:00,1.50,30.00,20.00,10.00,1
1,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1.0,FASTSHIP,EXPRESS,HUB_SCANNER,2026-07-12T03:30:00Z,...,INR,Ramesh Mohanty,9861012345,"Plot 42, Saheed Nagar, Bhubaneswar",NaT,1.50,30.00,20.00,10.00,1
2,SHP-10002,NaN,ACTIVE,TRK-9002,PCL-9002,1.0,ROADX,STANDARD,DRIVER_APP,2026-07-13 08:00,...,INR,Sujata Das,9437054321,"Link Road, Near Nishamani Hall, Cuttack",NaT,2.20,30.48,20.32,15.24,1
3,SHP-10003,NaN,ACTIVE,TRK-9003,PCL-9003,2.0,ROADX,STANDARD,HUB_SCANNER,2026-07-13T10:00:00Z,...,INR,Bikash Jena,8249011223,"Buxi Bazar, Main Road, Cuttack",NaT,NaN,40.00,30.00,20.00,1
4,SHP-10004,SHP-09950,RETURN_RTS,TRK-9002,PCL-9004,1.0,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-11T04:00:00Z,...,INR,Anita Patra,7008122334,"Kalpana Square, Laxmisagar, BBSR",NaT,2.00,25.00,25.00,25.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1.0,ODISHAEXPRESS,NEXTDAY,API_WEBHOOK,2026-07-10 10:54 +05:30,...,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 05:24:00+00:00,20.95,30.00,20.00,15.00,1
9996,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1.0,FASTSHIP,NEXTDAY,HUB_SCANNER,2026-07-10T12:51:00Z,...,NaN,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",NaT,20.95,30.00,20.00,15.00,1
9997,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1.0,ODISHAEXPRESS,NEXTDAY,DRIVER_APP,2026-07-11 00:04 +05:30,...,INR,Manas Bhoi,07110896587,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 18:34:00+00:00,20.95,30.00,20.00,15.00,1
9998,TEST-11916,NaN,TEST_ORDER,TRK-11916,PCL-11916,NaN,ODISHAEXPRESS,EXPRESS,DRIVER_APP,2026-07-23 10:43 +05:30,...,INR,NaN,9069150103,"Fakir Mohan Golai, Station Road, Balasore",2026-07-23 05:13:00+00:00,4.20,30.00,20.00,15.00,1


### 8. Standardize postal codes, country codes, and depot/location identifiers.

In [19]:
dest_map={
    'Bhubaneswar 751001': 'BBSR-751001',
    'Cuttack 753001': 'CTC-753001',
    'Puri 752001': 'PURI-752001',
    'Berhampur 760001': 'BAM-760001',
    'Bhad rak 756100': 'BHC-756100',
    'Rourkela 769001': 'RKL-769001',
    'Sambalpur 768001': 'SBP-768001',
    'Balasore 756001': 'BLS-756001',
    'UNKNOWN-999999':'UNKNOWN'
}
df['Destination_Zone']=df['Destination_Zone'].replace(dest_map)
df['Destination_Zone']

0       BBSR-751001
1       BBSR-751001
2        CTC-753001
3        CTC-753001
4           UNKNOWN
           ...     
9995     RKL-769001
9996     RKL-769001
9997     RKL-769001
9998     BAM-760001
9999     BAM-760001
Name: Destination_Zone, Length: 9962, dtype: object

### 9. Validate GPS latitude/longitude bounds and flag implausible jumps or missing coordinates.

In [20]:
df[['Lat', 'Lon']] = df['GPS_Coords'].str.split(',', expand=True).astype(float)

df['GPS_Valid'] = np.where(
    df['Lat'].between(-90, 90) & df['Lon'].between(-180, 180),
    1, 0
)

In [21]:
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Street_Address,Event_TS_UTC,Weight_kg,Length_cm,Width_cm,Height_cm,Dimensions_Valid,Lat,Lon,GPS_Valid
0,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1.0,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-12 09:00 +05:30,...,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-12 03:30:00+00:00,1.50,30.00,20.00,10.00,1,20.2961,85.8245,1
1,SHP-10001,NaN,ACTIVE,TRK-9001,PCL-9001,1.0,FASTSHIP,EXPRESS,HUB_SCANNER,2026-07-12T03:30:00Z,...,"Plot 42, Saheed Nagar, Bhubaneswar",NaT,1.50,30.00,20.00,10.00,1,20.2961,85.8245,1
2,SHP-10002,NaN,ACTIVE,TRK-9002,PCL-9002,1.0,ROADX,STANDARD,DRIVER_APP,2026-07-13 08:00,...,"Link Road, Near Nishamani Hall, Cuttack",NaT,2.20,30.48,20.32,15.24,1,20.4625,85.8830,1
3,SHP-10003,NaN,ACTIVE,TRK-9003,PCL-9003,2.0,ROADX,STANDARD,HUB_SCANNER,2026-07-13T10:00:00Z,...,"Buxi Bazar, Main Road, Cuttack",NaT,NaN,40.00,30.00,20.00,1,125.0000,-220.0000,0
4,SHP-10004,SHP-09950,RETURN_RTS,TRK-9002,PCL-9004,1.0,FASTSHIP,EXPRESS,DRIVER_APP,2026-07-11T04:00:00Z,...,"Kalpana Square, Laxmisagar, BBSR",NaT,2.00,25.00,25.00,25.00,1,20.2700,85.8400,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1.0,ODISHAEXPRESS,NEXTDAY,API_WEBHOOK,2026-07-10 10:54 +05:30,...,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 05:24:00+00:00,20.95,30.00,20.00,15.00,1,22.2578,84.8639,1
9996,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1.0,FASTSHIP,NEXTDAY,HUB_SCANNER,2026-07-10T12:51:00Z,...,"Plot 42, Saheed Nagar, Bhubaneswar",NaT,20.95,30.00,20.00,15.00,1,22.2702,84.8403,1
9997,SHP-11915,NaN,ACTIVE,TRK-11915,PCL-11915,1.0,ODISHAEXPRESS,NEXTDAY,DRIVER_APP,2026-07-11 00:04 +05:30,...,"Plot 42, Saheed Nagar, Bhubaneswar",2026-07-10 18:34:00+00:00,20.95,30.00,20.00,15.00,1,22.2333,84.8381,1
9998,TEST-11916,NaN,TEST_ORDER,TRK-11916,PCL-11916,NaN,ODISHAEXPRESS,EXPRESS,DRIVER_APP,2026-07-23 10:43 +05:30,...,"Fakir Mohan Golai, Station Road, Balasore",2026-07-23 05:13:00+00:00,4.20,30.00,20.00,15.00,1,19.3017,84.8115,1


### 10. Normalize route and stop sequence numbers; detect duplicate or skipped stops.

In [22]:
# 1. Normalize stop sequence numbers (replace zero or negative values with NaN)
df.loc[df['Stop_Seq'] <= 0, 'Stop_Seq'] = np.nan

# 2. Sort the dataset by Route and Stop Sequence to analyze the order
df = df.sort_values(by=['Route_ID', 'Stop_Seq'])

# 3. Detect duplicate stops (same stop number appearing more than once on the same route)
df['Duplicate_Stop_Flag'] = df.duplicated(subset=['Route_ID', 'Stop_Seq'], keep=False).astype(int)

# 4. Detect skipped stops (calculates the gap between consecutive stops on the same route)
df['Skipped_Stop_Flag'] = np.where(df.groupby('Route_ID')['Stop_Seq'].diff() > 1, 1, 0)

In [23]:
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Weight_kg,Length_cm,Width_cm,Height_cm,Dimensions_Valid,Lat,Lon,GPS_Valid,Duplicate_Stop_Flag,Skipped_Stop_Flag
4624,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,DRIVER_APP,2026-07-27T07:26:00Z,...,4.50,30.00,20.00,15.00,1,21.4900,83.9631,1,1,0
4625,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,API_WEBHOOK,2026-07-27T14:41:00Z,...,4.50,30.00,20.00,15.00,1,21.4619,83.9559,1,1,0
4626,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,HUB_SCANNER,2026-07-27T22:19:00Z,...,4.50,30.00,20.00,15.00,1,21.4787,84.0084,1,1,0
4627,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,BLUEJET,NEXTDAY,MANUAL_ENTRY,2026-07-28T03:40:00Z,...,4.50,30.48,20.32,15.24,1,NaN,NaN,0,1,0
4628,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,API_WEBHOOK,2026-07-28T06:55:00Z,...,4.50,30.00,20.00,15.00,1,21.4373,84.0011,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2777,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,DRIVER_APP,2026-07-25T15:44:00Z,...,7.44,30.00,20.00,15.00,1,19.8325,85.8324,1,1,1
2778,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,API_WEBHOOK,2026-07-26 04:29 +05:30,...,7.44,30.00,20.00,15.00,1,19.8169,85.8519,1,1,0
2779,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,API_WEBHOOK,2026-07-26T03:15:00Z,...,7.44,30.00,20.00,15.00,1,19.8374,85.8402,1,1,0
2780,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,MANUAL_ENTRY,2026-07-26T12:33:00Z,...,7.44,30.00,20.00,15.00,1,19.7927,85.8547,1,1,0


### **`Scenario 11: Proof-of-Delivery Validation`**
---

#### **Objective :-**
`Check whether delivery records have a shipment ID, POD ID,
and a valid delivery attempt number.`


**Code cell 1: Check POD links**

In [24]:
df["Attempt_No_Num"] = pd.to_numeric(
    df["Attempt_No"], errors="coerce"
)

pod_issues = df[
    df["POD_ID"].notna() &
    (
        df["Shipment_ID"].isna() |
        df["Attempt_No_Num"].isna() |
        (df["Attempt_No_Num"] < 0)
    )
]

print("POD linking issues:", len(pod_issues))

pod_issues[
    ["Shipment_ID", "Tracking_ID", "Attempt_No", "POD_ID", "Status"]
].head(10)

POD linking issues: 0


,Shipment_ID,Tracking_ID,Attempt_No,POD_ID,Status


**Code cell 2: Find delivery records without a POD ID**

In [25]:
status = df["Status"].fillna("").astype(str).str.upper()

delivery_rows = df[
    status.str.contains("DELIVER", na=False)
]

missing_pod = delivery_rows[
    delivery_rows["POD_ID"].isna()
]

print("Delivery rows without POD ID:", len(missing_pod))

missing_pod[
    ["Shipment_ID", "Tracking_ID", "Attempt_No", "Status", "POD_ID"]
].head(10)

Delivery rows without POD ID: 1628


,Shipment_ID,Tracking_ID,Attempt_No,Status,POD_ID
4691,SHP-10898,TRK-10898,0,OUT_FOR_DELIVERY,NaN
9238,SHP-11769,TRK-11769,0,OUT_FOR_DELIVERY,NaN
6594,SHP-11265,TRK-11265,0,OUT_FOR_DELIVERY,NaN
6466,SHP-11241,TRK-11241,0,OUT_FOR_DELIVERY,NaN
3356,SHP-10642,TRK-10642,0,OUT_FOR_DELIVERY,NaN
7487,SHP-11437,TRK-11437,0,OUT_FOR_DELIVERY,NaN
5778,SHP-11108,TRK-11108,0,OUT_FOR_DELIVERY,NaN
8763,SHP-11680,TRK-11680,0,OUT_FOR_DELIVERY,NaN
8682,SHP-11664,TRK-11664,0,OUT_FOR_DELIVERY,NaN
5074,SHP-10972,TRK-10972,0,OUT_FOR_DELIVERY,NaN


### **`Scenario 12: Normalize Exception and Failure Reason Codes`**
---

#### **Objective**
* Different labels may represent the same delivery problem.

* For example:
- cust_not_home
- Customer Unavailable
- Recipient Unavailable

* These labels should be standardized into one common value.

In [26]:
reason = (
    df["Exception_Reason"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

reason_map = {
    "cust_not_home": "CUSTOMER_UNAVAILABLE",
    "customer unavailable": "CUSTOMER_UNAVAILABLE",
    "recipient unavailable": "CUSTOMER_UNAVAILABLE",

    "bad_address": "ADDRESS_ISSUE",
    "address issue": "ADDRESS_ISSUE",
    "incomplete address": "ADDRESS_ISSUE",

    "crushed box": "DAMAGED_PARCEL",
    "damaged": "DAMAGED_PARCEL",
    "pkg_damaged": "DAMAGED_PARCEL",

    "weather": "WEATHER_DELAY",
    "heavy_rain": "WEATHER_DELAY",
    "cyclone alert": "WEATHER_DELAY"
}


df["Exception_Reason_Standard"] = reason.map(reason_map)

other_reason = (
    df["Exception_Reason"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", "_", regex=True)
    .str.strip("_")
)

df.loc[
    df["Exception_Reason_Standard"].isna() & (reason != ""),
    "Exception_Reason_Standard"
] = other_reason

df.loc[
    reason == "",
    "Exception_Reason_Standard"
] = pd.NA

df[
    ["Exception_Reason", "Exception_Reason_Standard"]
].drop_duplicates().head(20)

,Exception_Reason,Exception_Reason_Standard
4624,NaN,<NA>
7488,Crushed Box,DAMAGED_PARCEL
5779,Recipient Unavailable,CUSTOMER_UNAVAILABLE
5075,bad_address,ADDRESS_ISSUE
3580,Damaged,DAMAGED_PARCEL
5424,Weather,WEATHER_DELAY
2792,Incomplete Address,ADDRESS_ISSUE
57,Cyclone Alert,WEATHER_DELAY
7046,Address Issue,ADDRESS_ISSUE
9590,cust_not_home,CUSTOMER_UNAVAILABLE


### **`Scenario 13: Compare Promised and Actual Delivery Time`**
---

#### **Objective**

* Compare Promised_Delivery_TS with the timestamp of the
actual delivery event.


In [27]:
# Convert timestamps into datetime format
df["Event_TS_Parsed"] = pd.to_datetime(
    df["Event_TS"],
    errors="coerce",
    utc=True,
    format="mixed"
)

df["Promised_TS_Parsed"] = pd.to_datetime(
    df["Promised_Delivery_TS"],
    errors="coerce",
    utc=True,
    format="mixed"
)

status = (
    df["Status"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z]", "", regex=True)
)

delivery_events = df[
    status.str.contains("DELIVER", na=False)
].copy()

delivery_events["Delivery_Result"] = "UNKNOWN"

valid_times = (
    delivery_events["Event_TS_Parsed"].notna() &
    delivery_events["Promised_TS_Parsed"].notna()
)

delivery_events.loc[
    valid_times,
    "Delivery_Result"
] = "ON_TIME"

late = (
    valid_times &
    (
        delivery_events["Event_TS_Parsed"] >
        delivery_events["Promised_TS_Parsed"]
    )
)

delivery_events.loc[
    late,
    "Delivery_Result"
] = "LATE"

print(delivery_events["Delivery_Result"].value_counts())

delivery_events[
    [
        "Shipment_ID",
        "Tracking_ID",
        "Event_TS",
        "Promised_Delivery_TS",
        "Status",
        "Delivery_Result"
    ]
].head(10)

Delivery_Result
ON_TIME    1502
LATE       1236
UNKNOWN     141
Name: count, dtype: int64


,Shipment_ID,Tracking_ID,Event_TS,Promised_Delivery_TS,Status,Delivery_Result
4628,SHP-10886,TRK-10886,2026-07-28T06:55:00Z,2026-07-28T00:11:00Z,DELIVERED,LATE
4691,SHP-10898,TRK-10898,2026-07-19T18:10:00Z,2026-07-18T18:11:00Z,OUT_FOR_DELIVERY,LATE
4692,SHP-10898,TRK-10898,2026-07-20T10:34:00Z,2026-07-18T18:11:00Z,DELIVERED,LATE
1099,SHP-10212,TRK-10212,2026-07-24T02:40:00Z,2026-07-22T12:09:00Z,DELIVERED,LATE
9238,SHP-11769,TRK-11769,2026-07-02T14:12:00Z,2026-07-03T10:09:00Z,OUT_FOR_DELIVERY,ON_TIME
9239,SHP-11769,TRK-11769,2026-07-03T05:23:00Z,2026-07-03T10:09:00Z,DELIVERED,ON_TIME
6594,SHP-11265,TRK-11265,2026-07-19 06:01 +05:30,2026-07-19T13:14:00Z,OUT_FOR_DELIVERY,ON_TIME
6595,SHP-11265,TRK-11265,2026-07-19T07:51:00Z,2026-07-19T13:14:00Z,DELIVERED,ON_TIME
2551,SHP-10491,TRK-10491,2026-07-18 09:32 +05:30,2026-07-18T23:02:00Z,DELIVERED,ON_TIME
6466,SHP-11241,TRK-11241,2026-07-13 14:36 +05:30,2026-07-12T11:22:00Z,OUT_FOR_DELIVERY,LATE


#### **`Scenario 14: Identify Missing Pickup or Delivery Events`**
--- 

#### **Objective**
- Identify records with missing event timestamps and shipments without a pickup-like or delivery-like event in the available data.

**Code cell 1: Find missing timestamps**

In [28]:
missing_event_time = df[
    df["Event_TS_Parsed"].isna()
]

print("Rows with missing event timestamps:", len(missing_event_time))

missing_event_time[
    ["Shipment_ID", "Tracking_ID", "Event_Code", "Status", "Event_TS"]
].head(10)

Rows with missing event timestamps: 175


,Shipment_ID,Tracking_ID,Event_Code,Status,Event_TS
6592,SHP-11265,TRK-11265,PICKED_UP,PICKED_UP,NaN
720,SHP-10142,TRK-10142,PICKED_UP,PICKUP,NaN
3577,SHP-10686,TRK-10686,PICKED_UP,PICKED_UP,NaN
8236,SHP-11579,TRK-11579,CREATED,CREATED,NaN
6882,SHP-11321,TRK-11321,CREATED,LABEL_GENERATED,NaN
2853,SHP-10547,TRK-10547,DELIVERED,DELIV_SUCCESS,NaN
8308,SHP-11592,TRK-11592,IN_TRANSIT,IN_TRANSIT,NaN
9719,SHP-11861,TRK-11861,IN_TRANSIT,ARRIVED_AT_HUB,NaN
958,SHP-10187,TRK-10187,ORDER_CREATED,ORDER_CREATED,NaN
7905,SHP-11516,TRK-11516,PICKED_UP,PICKED_UP,NaN


**Code cell 2: Check pickup and delivery events by shipment**

In [29]:

status = df["Status"].fillna("").astype(str).str.upper()

df["Has_Pickup"] = status.str.contains("PICK", na=False)
df["Has_Delivery"] = status.str.contains("DELIVER", na=False)

shipment_events = df.groupby("Shipment_ID").agg(
    Has_Pickup=("Has_Pickup", "any"),
    Has_Delivery=("Has_Delivery", "any")
).reset_index()

shipment_events["Pickup_Check"] = shipment_events[
    "Has_Pickup"
].map({
    True: "FOUND",
    False: "UNKNOWN / NOT FOUND"
})

shipment_events["Delivery_Check"] = shipment_events[
    "Has_Delivery"
].map({
    True: "FOUND",
    False: "UNKNOWN / NOT FOUND"
})

shipment_events.head(10)

,Shipment_ID,Has_Pickup,Has_Delivery,Pickup_Check,Delivery_Check
0,SHP-10001,True,False,FOUND,UNKNOWN / NOT FOUND
1,SHP-10002,False,True,UNKNOWN / NOT FOUND,FOUND
2,SHP-10003,False,False,UNKNOWN / NOT FOUND,UNKNOWN / NOT FOUND
3,SHP-10004,False,True,UNKNOWN / NOT FOUND,FOUND
4,SHP-10005,True,True,FOUND,FOUND
5,SHP-10006,True,True,FOUND,FOUND
6,SHP-10007,True,True,FOUND,FOUND
7,SHP-10008,True,True,FOUND,FOUND
8,SHP-10009,True,True,FOUND,FOUND
9,SHP-10010,False,True,UNKNOWN / NOT FOUND,FOUND


### **`Scenario 15: Detect Possible Event Timestamp Ordering Issues`**

---

#### **Objective**
- Find cases where the event timestamp is earlier than the previous event timestamp for the same shipment in the original dataset row order.


In [30]:

events = df[
    df["Event_TS_Parsed"].notna()
].copy()

events["Previous_Event_TS"] = (
    events.groupby("Shipment_ID")["Event_TS_Parsed"].shift(1)
)

events["Time_Order_Issue"] = (
    events["Previous_Event_TS"].notna() &
    (
        events["Event_TS_Parsed"] <
        events["Previous_Event_TS"]
    )
)

time_issues = events[
    events["Time_Order_Issue"]
]

print("Possible timestamp ordering issues:", len(time_issues))

time_issues[
    [
        "Shipment_ID",
        "Previous_Event_TS",
        "Event_TS",
        "Event_Code",
        "Status"
    ]
].head(10)

Possible timestamp ordering issues: 284


,Shipment_ID,Previous_Event_TS,Event_TS,Event_Code,Status
217,SHP-10045,2026-07-11 11:34:00+00:00,2026-07-10 08:35 +05:30,PICKED_UP,PICKED_UP
6089,SHP-11169,2026-07-23 01:11:00+00:00,2026-07-22T06:42:00Z,PICKED_UP,PICKED_UP
3579,SHP-10686,2026-07-04 06:56:00+00:00,2026-07-03 21:22 +05:30,OUT_FOR_DELIVERY,OUT_FOR_DELIVERY
9443,SHP-11809,2026-07-09 03:42:00+00:00,2026-07-08 09:40 +05:30,PICKED_UP,PICKED_UP
9849,SHP-11886,2026-07-29 10:32:00+00:00,2026-07-27 23:09 +05:30,DELIVERY_SUCCESS,DELIVERED
2835,SHP-10544,2026-07-21 06:07:00+00:00,2026-07-20 13:35 +05:30,ARRIVED_AT_HUB,ARRIVED_AT_HUB
7094,SHP-11360,2026-07-12 08:49:00+00:00,2026-07-12T03:32:00Z,ARRIVED_AT_HUB,IN_TRANSIT
4765,SHP-10913,2026-07-11 14:07:00+00:00,2026-07-10 14:39,PICKED_UP,PICKUP
8305,SHP-11592,2026-07-13 17:44:00+00:00,2026-07-12T04:48:00Z,ORDER_CREATED,ORDER_CREATED
2608,SHP-10502,2026-07-09 18:12:00+00:00,2026-07-08 23:00 +05:30,LABEL_GENERATED,LABEL_GENERATED


In [9]:
# Standardize event codes
event_code = (
    df['Event_Code'].astype('string').str.upper().str.strip()
)

# Identify pickup and delivery events
pickup_events = df[
    event_code.isin(['PICKUP', 'PICKED_UP'])
].copy()

delivery_events = df[
    event_code.isin(['DELIVERED', 'DELIVERY_SUCCESS'])
].copy()

# Find first pickup timestamp
pickup_times = (
    pickup_events.groupby('Shipment_ID')['Event_TS_UTC']
    .min()
    .rename('First_Pickup')
)

# Find first delivery timestamp
delivery_times = (
    delivery_events.groupby('Shipment_ID')['Event_TS_UTC']
    .min()
    .rename('First_Delivery')
)

# Combine timestamps
event_check = pd.concat(
    [pickup_times, delivery_times],
    axis=1
).reset_index()

# Flag impossible ordering
event_check['Ordering_Issue'] = (
    event_check['First_Pickup'].notna()
    & event_check['First_Delivery'].notna()
    & (
        event_check['First_Delivery']
        < event_check['First_Pickup']
    )
)

# Calculate transit time only for valid ordering
event_check['Transit_Hours'] = (
    event_check['First_Delivery']
    - event_check['First_Pickup']
).dt.total_seconds() / 3600

event_check.loc[
    event_check['Ordering_Issue'],
    'Transit_Hours'
] = np.nan

display(event_check)

,Shipment_ID,First_Pickup,First_Delivery,Ordering_Issue,Transit_Hours
0,SHP-10001,2026-07-12 03:30:00+00:00,NaT,False,NaN
1,SHP-10003,NaT,NaT,False,NaN
2,SHP-10005,NaT,NaT,False,NaN
3,SHP-10006,2026-07-10 01:17:00+00:00,2026-07-10 22:58:00+00:00,False,21.683333
4,SHP-10007,NaT,NaT,False,NaN
...,...,...,...,...,...
1902,SHP-11849,NaT,NaT,False,NaN
1903,SHP-11871,NaT,2026-07-21 17:10:00+00:00,False,NaN
1904,SHP-11880,NaT,NaT,False,NaN
1905,SHP-11901,NaT,NaT,False,NaN


### **`Scenario 16: Exclude Test, Cancelled, and Voided Orders`**
---


#### **Objective**
- Exclude test, cancelled, and voided orders from standard business metrics while retaining the original dataset.

In [31]:
order_status = (
    df["Order_Status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

excluded_statuses = [
    "TEST_ORDER",
    "CANCELLED",
    "VOIDED"
]

excluded_orders = df[
    order_status.isin(excluded_statuses)
].copy()

analysis_df = df[
    ~order_status.isin(excluded_statuses)
].copy()

print("Original rows:", len(df))
print("Excluded rows:", len(excluded_orders))
print("Rows for analysis:", len(analysis_df))

excluded_orders["Order_Status"].value_counts()

Original rows: 9962
Excluded rows: 286
Rows for analysis: 9676


Order_Status
TEST_ORDER    141
VOIDED         76
CANCELLED      69
Name: count, dtype: int64

### **`Scenario 17: Validate Freight, Surcharge, COD, and Claim Amounts`**
---

#### **Objective**

1. Convert financial columns to numeric values.
2. Identify negative amounts.
3. Standardize currency values.



**Code cell 1: Find negative amounts**

In [32]:
amount_columns = [
    "Freight_Cost",
    "Surcharge",
    "COD_Amount",
    "Claim_Amount",
    "Declared_Value"
]

for column in amount_columns:
    df[column + "_Num"] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

negative_amounts = df[
    (df[[column + "_Num" for column in amount_columns]] < 0)
    .any(axis=1)
]

print("Rows with negative amounts:", len(negative_amounts))

negative_amounts[
    ["Shipment_ID"] + amount_columns
].head(10)

Rows with negative amounts: 0


,Shipment_ID,Freight_Cost,Surcharge,COD_Amount,Claim_Amount,Declared_Value


**Code cell 2: Standardize currency**

In [33]:
currency = (
    df["Currency"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

currency_map = {
    "INR": "INR",
    "RS": "INR",
    "₹": "INR"
}

df["Currency_Standard"] = currency.map(currency_map)

df.loc[
    (currency != "") & df["Currency_Standard"].isna(),
    "Currency_Standard"
] = currency

print(df["Currency_Standard"].value_counts(dropna=False))

Currency_Standard
INR    9758
NaN     204
Name: count, dtype: int64


#### **`scenario 18: Link Return Shipments to Parent Shipments`**
---

#### **Objective**

- Check whether Parent_Shipment_ID values exist in the Shipment_ID column.


In [34]:

known_shipments = set(
    df["Shipment_ID"].dropna().astype(str).str.strip()
)

parent_ids = (
    df["Parent_Shipment_ID"]
    .fillna("")
    .astype(str)
    .str.strip()
)

has_parent = parent_ids != ""

missing_parent_links = df[
    has_parent &
    ~parent_ids.isin(known_shipments)
]

print("Missing parent shipment links:", len(missing_parent_links))

missing_parent_links[
    ["Shipment_ID", "Parent_Shipment_ID", "Order_Status"]
].head(10)

Missing parent shipment links: 155


,Shipment_ID,Parent_Shipment_ID,Order_Status
8675,SHP-11663,SHP-INVALID-999,RETURN_RTS
8676,SHP-11663,SHP-INVALID-999,RETURN_RTS
8677,SHP-11663,SHP-INVALID-999,RETURN_RTS
8678,SHP-11663,SHP-INVALID-999,RETURN_RTS
4,SHP-10004,SHP-09950,RETURN_RTS
2849,SHP-10547,SHP-INVALID-999,RETURN_RTS
2850,SHP-10547,SHP-INVALID-999,RETURN_RTS
2851,SHP-10547,SHP-INVALID-999,RETURN_RTS
2853,SHP-10547,SHP-INVALID-999,RETURN_RTS
2852,SHP-10547,SHP-INVALID-999,RETURN_RTS


### **`Scenario 19: Mask Customer Personal Information`**
---

#### **Objective**

- Mask customer names, phone numbers, and street addresses before sharing data in reports or public GitHub repositories.


**Code cell 1: Mask personal information**

In [35]:
safe_df = df.copy()

personal_columns = [
    "Customer_Name",
    "Customer_Phone",
    "Street_Address"
]

for column in personal_columns:

    safe_df[column] = safe_df[column].apply(
        lambda value: "MASKED" if pd.notna(value) else "MISSING"
    )

safe_df[
    ["Customer_Name", "Customer_Phone", "Street_Address"]
].head(10)

,Customer_Name,Customer_Phone,Street_Address
4624,MASKED,MASKED,MASKED
4625,MASKED,MASKED,MASKED
4626,MASKED,MASKED,MASKED
4627,MASKED,MASKED,MASKED
4628,MASKED,MASKED,MASKED
4688,MASKED,MASKED,MASKED
4689,MASKED,MASKED,MASKED
4690,MASKED,MASKED,MASKED
4691,MASKED,MASKED,MASKED
4692,MASKED,MASKED,MASKED


**Code cell 2: Save the masked dataset**

In [36]:
safe_df.to_csv(
    "shipments_masked.csv",
    index=False
)

print("Masked dataset saved successfully.")

Masked dataset saved successfully.


#### **`Scenario 20: Duplicate Tracking IDs and Unresolved Locations`**
---


#### **Objective**
1. Identify tracking IDs connected to multiple active shipments.
2. Find missing or unresolved origin depots.
3. Find missing or unresolved destination zones.

**Code cell 1: Find tracking IDs linked to multiple active shipments**

In [37]:
order_status = (
    df["Order_Status"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

active_df = df[
    order_status == "ACTIVE"
].copy()

tracking_check = (
    active_df.groupby("Tracking_ID")["Shipment_ID"]
    .nunique()
    .reset_index()
)

tracking_check.columns = [
    "Tracking_ID",
    "Shipment_Count"
]

duplicate_tracking = tracking_check[
    tracking_check["Shipment_Count"] > 1
]

print(
    "Tracking IDs linked to multiple active shipments:",
    len(duplicate_tracking)
)

duplicate_tracking.head(10)

Tracking IDs linked to multiple active shipments: 24


,Tracking_ID,Shipment_Count
13,TRK-10020,2
15,TRK-10022,2
28,TRK-10037,2
95,TRK-10110,2
148,TRK-10168,2
198,TRK-10220,2
274,TRK-10305,2
305,TRK-10339,2
531,TRK-10580,2
622,TRK-10680,2


**Code cell 2: Find unresolved origin and destination values**

In [38]:
def location_needs_review(value):

    if pd.isna(value):
        return True

    value = str(value).strip().upper()

    if value == "":
        return True

    if (
        "UNKNOWN" in value or
        "TBD" in value or
        "UNMAPPED" in value
    ):
        return True

    return False


bad_origin = df[
    df["Origin_Depot"].apply(location_needs_review)
]

bad_destination = df[
    df["Destination_Zone"].apply(location_needs_review)
]

print("Origin locations needing review:", len(bad_origin))
print("Destination locations needing review:", len(bad_destination))

print("\nOrigin depot values:")
print(bad_origin["Origin_Depot"].value_counts().head(10))

print("\nDestination zone values:")
print(bad_destination["Destination_Zone"].value_counts().head(10))

Origin locations needing review: 0
Destination locations needing review: 1115

Origin depot values:
Series([], Name: count, dtype: int64)

Destination zone values:
Destination_Zone
UNKNOWN     581
ZONE-TBD    534
Name: count, dtype: int64


### **`Final Step: Save Your Processed Dataset`**
---


- Save the processed dataset separately from the original file.

- The original dataset should remain unchanged so that the cleaning and validation results can be reviewed later.

In [39]:
# Save the dataset with the added validation columns
df.to_csv(
    "shipments_tested.csv",
    index=False
)

print("Tested dataset saved successfully.")
print("Original dataset rows:", len(df))
print("Tested dataset rows:", len(df)) 

Tested dataset saved successfully.
Original dataset rows: 9962
Tested dataset rows: 9962


In [40]:
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Event_TS_Parsed,Promised_TS_Parsed,Has_Pickup,Has_Delivery,Freight_Cost_Num,Surcharge_Num,COD_Amount_Num,Claim_Amount_Num,Declared_Value_Num,Currency_Standard
4624,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,DRIVER_APP,2026-07-27T07:26:00Z,...,2026-07-27 07:26:00+00:00,2026-07-28 00:11:00+00:00,False,False,0.00,15.0,0.899,0,12094.88,INR
4625,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,API_WEBHOOK,2026-07-27T14:41:00Z,...,2026-07-27 14:41:00+00:00,2026-07-28 00:11:00+00:00,True,False,373.35,15.0,899.000,0,12094.88,INR
4626,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,HUB_SCANNER,2026-07-27T22:19:00Z,...,2026-07-27 22:19:00+00:00,2026-07-28 00:11:00+00:00,False,False,373.35,15.0,899.000,0,12094.88,INR
4627,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,BLUEJET,NEXTDAY,MANUAL_ENTRY,2026-07-28T03:40:00Z,...,2026-07-28 03:40:00+00:00,2026-07-28 00:11:00+00:00,False,False,373.35,15.0,899.000,0,12094.88,INR
4628,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,API_WEBHOOK,2026-07-28T06:55:00Z,...,2026-07-28 06:55:00+00:00,2026-07-28 00:11:00+00:00,False,True,0.00,15.0,899.000,0,12094.88,INR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2777,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,DRIVER_APP,2026-07-25T15:44:00Z,...,2026-07-25 15:44:00+00:00,2026-07-27 10:25:00+00:00,False,False,295.33,25.0,450.000,0,3030.18,INR
2778,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,API_WEBHOOK,2026-07-26 04:29 +05:30,...,2026-07-25 22:59:00+00:00,2026-07-27 10:25:00+00:00,True,False,295.33,25.0,450.000,0,3030.18,INR
2779,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,API_WEBHOOK,2026-07-26T03:15:00Z,...,2026-07-26 03:15:00+00:00,2026-07-27 10:25:00+00:00,False,False,295.33,25.0,450.000,0,3030.18,INR
2780,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,MANUAL_ENTRY,2026-07-26T12:33:00Z,...,2026-07-26 12:33:00+00:00,2026-07-27 10:25:00+00:00,False,True,295.33,25.0,450.000,0,3030.18,INR


## **Transformation Scenarios**
---

### **`Scenario 1. Calculate daily shipment volume by origin, destination zone, carrier, and service level.`**

In [41]:
daily_vol = df.groupby([
    df['Event_TS_UTC'].dt.date,
    'Origin_Depot',
    'Destination_Zone',
    'Carrier',
    'Service_Level'
])['Tracking_ID'].count().reset_index(name='Shipment_Volume')

In [42]:
daily_vol

,Event_TS_UTC,Origin_Depot,Destination_Zone,Carrier,Service_Level,Shipment_Volume
0,2026-07-01,EAST,RKL-769001,FASTSHIP,STANDARD,1
1,2026-07-01,EAST,RKL-769001,ODISHAEXPRESS,STANDARD,1
2,2026-07-01,NORTH,BBSR-751001,BLUEJET,STANDARD,1
3,2026-07-01,NORTH,BLS-756001,ODISHAEXPRESS,STANDARD,1
4,2026-07-01,NORTH,UNKNOWN,ODISHAEXPRESS,SURFACE,1
...,...,...,...,...,...,...
2672,2026-07-31,SOUTH,RKL-769001,ROADX,EXPRESS,2
2673,2026-07-31,SOUTH,SBP-768001,BLUEJET,ECONOMY,2
2674,2026-07-31,SOUTH,UNKNOWN,ROADX,EXPRESS,1
2675,2026-07-31,WEST,BAM-760001,FASTSHIP,ECONOMY,1


### **`Scenario 2. Measure on-time pickup, on-time delivery, and delivery-in-full rates against promised windows.`**

In [43]:
# Isolate the delivery events to measure final fulfillment performance
deliveries = df[df['Event_Code'] == 'DELIVERED'].copy()

# Flag On-Time Delivery (1 if delivered before/on promised time, 0 if late)
deliveries['OTD_Flag'] = np.where(deliveries['Event_TS_UTC'] <= deliveries['Promised_Delivery_TS'], 1, 0)

# Flag Delivery-in-Full (1 if delivered with no exceptions or damage, 0 otherwise)
deliveries['DIF_Flag'] = np.where(deliveries['Exception_Reason'].isna(), 1, 0)

# Group by Carrier to calculate the overall percentage rates (mean of the 0/1 flags)
performance_rates = deliveries.groupby('Carrier').agg(
    Total_Deliveries=('Tracking_ID', 'count'),
    OTD_Rate=('OTD_Flag', 'mean'),
    DIF_Rate=('DIF_Flag', 'mean')
).reset_index()

# Convert the decimal rates to standard percentages
performance_rates['OTD_Rate'] = performance_rates['OTD_Rate'] * 100
performance_rates['DIF_Rate'] = performance_rates['DIF_Rate'] * 100

In [44]:
performance_rates

,Carrier,Total_Deliveries,OTD_Rate,DIF_Rate
0,BLUEJET,302,15.562914,100.000000
1,FASTSHIP,361,14.681440,99.722992
2,ODISHAEXPRESS,263,17.490494,100.000000
3,ROADX,403,15.880893,100.000000


### **`Scenario 3. Calculate transit time and warehouse dwell time between defined scan milestones.`**

In [45]:
event_mapping = {
    'ORDER_CREATED': 'CREATED', 'LABEL_GENERATED': 'CREATED', 'CREATED': 'CREATED',
    'PICKED_UP': 'PICKUP', 'PICKUP': 'PICKUP',
    'DELIVERY_SUCCESS': 'DELIVERED', 'DELIVERED': 'DELIVERED',
    'DELIVERY_FAILED': 'EXCEPTION', 'HOLD': 'EXCEPTION', 'EXCEPTION': 'EXCEPTION'
}

df['Event_Code'] = df['Event_Code'].str.upper().replace(event_mapping)

milestones = df.pivot_table(index='Tracking_ID', columns='Event_Code', values='Event_TS_UTC', aggfunc='min')

milestones['Transit_Time_Hours'] = (milestones['DELIVERED'] - milestones['PICKUP']).dt.total_seconds() / 3600.0
milestones['Warehouse_Dwell_Hours'] = (milestones['PICKUP'] - milestones['CREATED']).dt.total_seconds() / 3600.0

milestones['Transit_Time_Hours'] = milestones['Transit_Time_Hours'].round(2)
milestones['Warehouse_Dwell_Hours'] = milestones['Warehouse_Dwell_Hours'].round(2)

In [46]:
milestones

Event_Code,ARRIVED_AT_HUB,CREATED,DELIVERED,EXCEPTION,IN_TRANSIT,OUT_FOR_DELIVERY,OUT_FOR_DELVERY,PICKUP,RETURNED_TO_SENDER,Transit_Time_Hours,Warehouse_Dwell_Hours
Tracking_ID,,,,,,,,,,,
TRK-10005,NaT,2026-07-23 09:15:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaN,NaN
TRK-10006,NaT,NaT,2026-07-10 22:58:00+00:00,NaT,NaT,NaT,NaT,2026-07-10 01:17:00+00:00,NaT,21.68,NaN
TRK-10007,NaT,NaT,NaT,NaT,2026-07-28 13:40:00+00:00,2026-07-28 20:34:00+00:00,NaT,NaT,NaT,NaN,NaN
TRK-10008,NaT,NaT,NaT,NaT,NaT,2026-07-24 23:46:00+00:00,NaT,NaT,NaT,NaN,NaN
TRK-10009,NaT,NaT,2026-07-07 04:00:00+00:00,NaT,NaT,2026-07-06 12:45:00+00:00,NaT,NaT,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
TRK-11913,NaT,2026-07-24 22:41:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaN,NaN
TRK-11914,NaT,2026-07-06 21:07:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaN,NaN
TRK-11915,NaT,NaT,2026-07-10 18:34:00+00:00,NaT,2026-07-10 05:24:00+00:00,NaT,NaT,NaT,NaT,NaN,NaN


### **`Scenario 4. Build first-attempt delivery success and repeat-attempt metrics.`**

In [47]:
# Isolate only the final delivered events to evaluate how many attempts it took
delivered_df = df[df['Event_Code'] == 'DELIVERED'].copy()

# Create binary flags: 1 if delivered on the first attempt, 0 if not
delivered_df['First_Attempt'] = np.where(delivered_df['Attempt_No'] <= 1, 1, 0)

# Create binary flags: 1 if it took more than one attempt, 0 if not
delivered_df['Repeat_Attempt'] = np.where(delivered_df['Attempt_No'] > 1, 1, 0)

# Group by Carrier to calculate the overall percentage rates for these metrics
attempt_metrics = delivered_df.groupby('Carrier').agg(
    Total_Deliveries=('Tracking_ID', 'count'),
    First_Attempt_Rate=('First_Attempt', 'mean'),
    Repeat_Attempt_Rate=('Repeat_Attempt', 'mean')
).reset_index()

# Convert the decimal rates to standard percentages and round to 2 decimal places
attempt_metrics['First_Attempt_Rate'] = (attempt_metrics['First_Attempt_Rate'] * 100).round(2)
attempt_metrics['Repeat_Attempt_Rate'] = (attempt_metrics['Repeat_Attempt_Rate'] * 100).round(2)

In [48]:
attempt_metrics

,Carrier,Total_Deliveries,First_Attempt_Rate,Repeat_Attempt_Rate
0,BLUEJET,408,94.36,5.64
1,FASTSHIP,498,94.18,5.82
2,ODISHAEXPRESS,373,95.71,4.29
3,ROADX,524,95.61,4.39


### **`Scenario 5. Measure exception rates by reason, carrier, depot, and delivery zone.`**

In [49]:
# 1. Calculate the total shipment volume for every combination of Carrier, Depot, and Zone
total_volume = df.groupby(['Carrier', 'Origin_Depot', 'Destination_Zone'])['Tracking_ID'].count().reset_index(name='Total_Volume')

# 2. Isolate only the rows where an exception occurred and count them by the 4 requested dimensions
exceptions_only = df[df['Exception_Reason'].notna()]
exception_counts = exceptions_only.groupby(['Exception_Reason', 'Carrier', 'Origin_Depot', 'Destination_Zone'])['Tracking_ID'].count().reset_index(name='Exception_Count')

# 3. Merge the two datasets together to calculate the specific exception rate percentage
exception_rates = pd.merge(exception_counts, total_volume, on=['Carrier', 'Origin_Depot', 'Destination_Zone'], how='inner')
exception_rates['Exception_Rate_%'] = ((exception_rates['Exception_Count'] / exception_rates['Total_Volume']) * 100).round(2)

In [50]:
exception_rates.head(10)

,Exception_Reason,Carrier,Origin_Depot,Destination_Zone,Exception_Count,Total_Volume,Exception_Rate_%
0,Address Issue,BLUEJET,EAST,BLS-756001,1,39,2.56
1,Address Issue,BLUEJET,EAST,RKL-769001,1,56,1.79
2,Address Issue,BLUEJET,NORTH,BLS-756001,1,73,1.37
3,Address Issue,BLUEJET,SOUTH,UNKNOWN,1,32,3.12
4,Address Issue,FASTSHIP,EAST,RKL-769001,2,108,1.85
5,Address Issue,FASTSHIP,NORTH,753001-CTC,1,38,2.63
6,Address Issue,FASTSHIP,SOUTH,BLS-756001,1,65,1.54
7,Address Issue,FASTSHIP,SOUTH,UNKNOWN,1,26,3.85
8,Address Issue,FASTSHIP,WEST,751001-BBSR,1,23,4.35
9,Address Issue,ODISHAEXPRESS,EAST,751001-BBSR,1,33,3.03


### **`Scenario 6. Calculate route adherence and stop completion rates where planned routes are available.`**

In [51]:
# 1. Filter out records where Route_ID or Stop_Seq is missing to analyze valid routes
routes_df = df.dropna(subset=['Route_ID', 'Stop_Seq']).copy()

# 2. Calculate the highest stop number for each route (serving as the total planned stops)
planned_stops = routes_df.groupby('Route_ID')['Stop_Seq'].max().reset_index(name='Total_Planned_Stops')

# 3. Count the actual number of unique stops successfully scanned on each route
visited_stops = routes_df.groupby('Route_ID')['Stop_Seq'].nunique().reset_index(name='Unique_Stops_Visited')

# 4. Merge the data and calculate the completion rate percentage
route_adherence = pd.merge(planned_stops, visited_stops, on='Route_ID')
route_adherence['Stop_Completion_Rate_%'] = ((route_adherence['Unique_Stops_Visited'] / route_adherence['Total_Planned_Stops']) * 100).round(2)

# 5. Cap the maximum rate at 100% (to handle any data anomalies where dirty data has more distinct stops than the max sequence)
route_adherence['Stop_Completion_Rate_%'] = np.where(route_adherence['Stop_Completion_Rate_%'] > 100, 100.0, route_adherence['Stop_Completion_Rate_%'])

In [52]:
route_adherence.head(15)

,Route_ID,Total_Planned_Stops,Unique_Stops_Visited,Stop_Completion_Rate_%
0,RT-100,36.0,7,19.44
1,RT-101,39.0,7,17.95
2,RT-102,36.0,5,13.89
3,RT-103,37.0,6,16.22
4,RT-104,36.0,4,11.11
5,RT-105,35.0,5,14.29
6,RT-106,39.0,8,20.51
7,RT-107,31.0,2,6.45
8,RT-108,43.0,6,13.95
9,RT-109,44.0,5,11.36


### **`Scenario 7. Summarize shipment cost, surcharge, claim cost, and cost per delivered parcel.`**

In [53]:
# List of financial columns to clean
financial_cols = ['Freight_Cost', 'Surcharge', 'COD_Amount', 'Claim_Amount']

for col in financial_cols:
    if col in df.columns:
        # Remove all non-numeric characters (keep only digits and decimals)
        df[col] = df[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
        
        # Convert the cleaned string to a float and replace any NaN values with 0.0
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

In [54]:
# 1. Aggregate the total financial components grouped by Carrier
cost_summary = df.groupby('Carrier').agg(
    Total_Freight=('Freight_Cost', 'sum'),
    Total_Surcharge=('Surcharge', 'sum'),
    Total_Claim=('Claim_Amount', 'sum')
).reset_index()

# 2. Count the number of successfully delivered unique shipments per Carrier
delivered_volume = df[df['Event_Code'] == 'DELIVERED'].groupby('Carrier')['Tracking_ID'].nunique().reset_index(name='Delivered_Parcels')

# 3. Merge the financial totals with the delivered volume
financial_summary = pd.merge(cost_summary, delivered_volume, on='Carrier', how='left')

# 4. Calculate the overall Total Cost and the Cost per Delivered Parcel
financial_summary['Total_Cost'] = financial_summary['Total_Freight'] + financial_summary['Total_Surcharge'] + financial_summary['Total_Claim']
financial_summary['Cost_Per_Delivered_Parcel'] = (financial_summary['Total_Cost'] / financial_summary['Delivered_Parcels']).round(2)

In [55]:
cost_summary

,Carrier,Total_Freight,Total_Surcharge,Total_Claim
0,BLUEJET,406550.82,40710.0,1500
1,FASTSHIP,500489.31,48405.0,5400
2,ODISHAEXPRESS,380994.27,37360.0,4600
3,ROADX,514096.28,48950.0,9500


In [56]:
delivered_volume

,Carrier,Delivered_Parcels
0,BLUEJET,392
1,FASTSHIP,476
2,ODISHAEXPRESS,355
3,ROADX,506


In [57]:
financial_summary

,Carrier,Total_Freight,Total_Surcharge,Total_Claim,Delivered_Parcels,Total_Cost,Cost_Per_Delivered_Parcel
0,BLUEJET,406550.82,40710.0,1500,392,448760.82,1144.80
1,FASTSHIP,500489.31,48405.0,5400,476,554294.31,1164.48
2,ODISHAEXPRESS,380994.27,37360.0,4600,355,422954.27,1191.42
3,ROADX,514096.28,48950.0,9500,506,572546.28,1131.51


### **`Scenario 8. Build volume and delay heatmaps by depot and time window.`**

In [58]:
# 1. Bring the calculated Dwell Hours from the milestones table back into the main df
df['Warehouse_Dwell_Hours'] = df['Tracking_ID'].map(milestones['Warehouse_Dwell_Hours'])

# 2. Extract the hour of the day from the event timestamp to serve as our time window
df['Time_Window_Hour'] = df['Event_TS_UTC'].dt.hour

# 3. Build the Volume Heatmap dataset (Count of unique shipments by Depot and Hour)
volume_heatmap = df.pivot_table(
    index='Time_Window_Hour', 
    columns='Origin_Depot', 
    values='Tracking_ID', 
    aggfunc='nunique', 
    fill_value=0
)

# 4. Build the Delay Heatmap dataset (Average dwell time by Depot and Hour)
delay_heatmap = df.pivot_table(
    index='Time_Window_Hour', 
    columns='Origin_Depot', 
    values='Warehouse_Dwell_Hours', 
    aggfunc='mean', 
    fill_value=0
).round(2)

In [59]:
volume_heatmap

Origin_Depot,EAST,NORTH,SOUTH,WEST
Time_Window_Hour,,,,
0.0,41,37,24,33
1.0,39,26,33,25
2.0,41,33,34,30
3.0,33,31,33,25
4.0,40,41,31,43
5.0,35,32,29,46
6.0,34,21,33,33
7.0,42,21,33,23
8.0,35,30,44,33


In [60]:
delay_heatmap

Origin_Depot,EAST,NORTH,SOUTH,WEST
Time_Window_Hour,,,,
0.0,11.10,11.75,7.95,12.80
1.0,10.91,12.51,8.40,6.68
2.0,9.45,9.56,9.53,10.96
3.0,15.62,14.00,11.81,10.27
4.0,-14.45,10.09,8.90,11.44
5.0,10.28,13.70,7.00,10.11
6.0,11.69,6.80,12.38,10.23
7.0,9.55,12.69,9.66,11.45
8.0,10.02,9.79,11.99,11.51


## **Scenario 9 : Measure scan compliance and missing-event rates by carrier.**

In [ ]:
#Checking which carriers have missing pickup, transit, or delivery events.
df['Event_Code'] = df['Event_Code'].astype('string').str.upper().str.strip()

df['Pickup'] = df['Event_Code'].isin(['PICKUP', 'PICKED_UP'])
df['Transit'] = df['Event_Code'].eq('IN_TRANSIT')
df['Delivery'] = df['Event_Code'].eq('DELIVERED')

scan_summary = df.groupby(['Carrier', 'Shipment_ID']).agg(
    Pickup=('Pickup', 'any'),
    Transit=('Transit', 'any'),
    Delivery=('Delivery', 'any')
).reset_index()

scan_compliance = scan_summary.groupby('Carrier').agg(
    Total_Shipments=('Shipment_ID', 'nunique'),
    Pickup_Completed=('Pickup', 'sum'),
    Transit_Completed=('Transit', 'sum'),
    Delivery_Completed=('Delivery', 'sum')
).reset_index()

scan_compliance['Missing_Pickup'] = (
    scan_compliance['Total_Shipments'] - scan_compliance['Pickup_Completed']
)

scan_compliance['Missing_Delivery'] = (
    scan_compliance['Total_Shipments'] - scan_compliance['Delivery_Completed']
)

display(scan_compliance)

,Carrier,Total_Shipments,Pickup_Completed,Transit_Completed,Delivery_Completed,Missing_Pickup,Missing_Delivery
0,BLUEJET,704,393,319,393,311,311
1,FASTSHIP,858,505,377,476,353,382
2,ODISHAEXPRESS,692,398,301,357,294,335
3,ROADX,874,521,400,506,353,368


## **Scenario 10 : Calculate damage, loss, and return-to-sender rates.**

In [62]:
#Identifying damaged, lost, and returned shipments.

status = df['Status'].fillna('').astype(str).str.upper()
reason = df['Exception_Reason'].fillna('').astype(str).str.upper()

df['Damaged'] = reason.str.contains('DAMAG', na=False)

df['Lost'] = (
    status.isin(['LOST', 'MISSING'])
    | reason.str.contains('LOST|LOSS', na=False, regex=True)
)

df['Returned'] = status.isin(['RETURNED', 'RETURN_TO_SENDER'])

outcomes = df.groupby('Shipment_ID').agg(
    Carrier=('Carrier', 'first'),
    Damaged=('Damaged', 'max'),
    Lost=('Lost', 'max'),
    Returned=('Returned', 'max')
).reset_index()

summary = outcomes.groupby('Carrier').agg(
    Total_Shipments=('Shipment_ID', 'nunique'),
    Damaged=('Damaged', 'sum'),
    Lost=('Lost', 'sum'),
    Returned=('Returned', 'sum')
).reset_index()

for col in ['Damaged', 'Lost', 'Returned']:
    summary[col + '_Rate'] = (
        summary[col] / summary['Total_Shipments'] * 100
    )

display(summary)

,Carrier,Total_Shipments,Damaged,Lost,Returned,Damaged_Rate,Lost_Rate,Returned_Rate
0,BLUEJET,437,7,0,10,1.601831,0.0,2.288330
1,FASTSHIP,540,9,0,15,1.666667,0.0,2.777778
2,ODISHAEXPRESS,400,8,0,8,2.000000,0.0,2.000000
3,ROADX,539,13,0,10,2.411874,0.0,1.855288


## **Scenario 11 : Create shipment journey timelines by joining standardized scan events.**

In [63]:
#Display the sequence of events for each shipment.

df['Event_TS_UTC'] = pd.to_datetime(
    df['Event_TS_UTC'], errors='coerce', utc=True
)

journey = df.sort_values(
    ['Shipment_ID', 'Event_TS_UTC']
).copy()

journey['Journey_Event'] = (
    journey['Event_TS_UTC'].astype('string')
    + ' - '
    + journey['Event_Code'].astype('string')
    + ' - '
    + journey['Status'].astype('string')
)

shipment_journey = journey.groupby('Shipment_ID').agg(
    Carrier=('Carrier', 'first'),
    First_Event=('Event_TS_UTC', 'min'),
    Last_Event=('Event_TS_UTC', 'max'),
    Event_Count=('Event_Code', 'count'),
    Journey=('Journey_Event', lambda x: ' -> '.join(x.dropna()))
).reset_index()

display(shipment_journey.head(10))

,Shipment_ID,Carrier,First_Event,Last_Event,Event_Count,Journey
0,SHP-10001,FASTSHIP,2026-07-12 03:30:00+00:00,2026-07-12 03:30:00+00:00,2,2026-07-12 03:30:00+00:00 - PICKUP - PICKED_UP
1,SHP-10002,ROADX,NaT,NaT,1,
2,SHP-10003,ROADX,NaT,NaT,1,
3,SHP-10004,FASTSHIP,NaT,NaT,1,
4,SHP-10005,BLUEJET,2026-07-23 09:15:00+00:00,2026-07-23 09:15:00+00:00,6,2026-07-23 09:15:00+00:00 - CREATED - CREATED
5,SHP-10006,FASTSHIP,2026-07-10 01:17:00+00:00,2026-07-10 22:58:00+00:00,6,2026-07-10 01:17:00+00:00 - PICKUP - PICKUP ->...
6,SHP-10007,ROADX,2026-07-28 13:40:00+00:00,2026-07-28 20:34:00+00:00,5,2026-07-28 13:40:00+00:00 - IN_TRANSIT - IN_TR...
7,SHP-10008,ROADX,2026-07-24 23:46:00+00:00,2026-07-24 23:46:00+00:00,5,2026-07-24 23:46:00+00:00 - OUT_FOR_DELIVERY -...
8,SHP-10009,ODISHAEXPRESS,2026-07-06 12:45:00+00:00,2026-07-07 04:00:00+00:00,5,2026-07-06 12:45:00+00:00 - OUT_FOR_DELIVERY -...
9,SHP-10010,FASTSHIP,2026-07-08 23:22:00+00:00,2026-07-09 17:31:00+00:00,5,2026-07-08 23:22:00+00:00 - ARRIVED_AT_HUB - I...


## **Scenario 12 : Forecast parcel volume by depot and weekday using historical data.**

In [64]:
#Estimate shipment volume by depot and weekday.

df['Event_TS_UTC'] = pd.to_datetime(
    df['Event_TS_UTC'], errors='coerce', utc=True
)

temp = df.dropna(
    subset=['Event_TS_UTC', 'Shipment_ID', 'Origin_Depot']
).copy()

temp['Date'] = temp['Event_TS_UTC'].dt.date
temp['Weekday'] = temp['Event_TS_UTC'].dt.day_name()

daily_volume = temp.groupby(
    ['Date', 'Origin_Depot', 'Weekday']
)['Shipment_ID'].nunique().reset_index(name='Volume')

forecast = daily_volume.groupby(
    ['Origin_Depot', 'Weekday']
).agg(
    Average_Volume=('Volume', 'mean'),
    Historical_Days=('Date', 'nunique')
).reset_index()

forecast['Forecast_Volume'] = forecast['Average_Volume'].round()

display(forecast)

,Origin_Depot,Weekday,Average_Volume,Historical_Days,Forecast_Volume
0,EAST,Friday,20.40,5,20.0
1,EAST,Monday,24.50,4,24.0
2,EAST,Saturday,29.25,4,29.0
3,EAST,Sunday,22.50,4,22.0
4,EAST,Thursday,19.80,5,20.0
5,EAST,Tuesday,24.75,4,25.0
6,EAST,Wednesday,20.40,5,20.0
7,NORTH,Friday,24.25,4,24.0
8,NORTH,Monday,23.75,4,24.0
9,NORTH,Saturday,27.25,4,27.0


## **Scenario 13 : Detect outlier transit times, GPS jumps, and sudden exception spikes.**

In [66]:
# Find unusual transit times and invalid GPS coordinates.

# Validate GPS coordinates
gps = df['GPS_Coords'].astype('string').str.extract(
    r'^\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*$'
)

df['Latitude'] = pd.to_numeric(gps[0], errors='coerce')
df['Longitude'] = pd.to_numeric(gps[1], errors='coerce')

df['Invalid_GPS'] = (
    ~df['Latitude'].between(-90, 90)
    | ~df['Longitude'].between(-180, 180)
)

print("Invalid GPS records:", df['Invalid_GPS'].sum())

# Detect outliers in warehouse dwell time
if 'Warehouse_Dwell_Hours' in df.columns:
    dwell = pd.to_numeric(
        df['Warehouse_Dwell_Hours'], errors='coerce'
    )

    q1 = dwell.quantile(0.25)
    q3 = dwell.quantile(0.75)
    iqr = q3 - q1

    df['Dwell_Outlier'] = (
        (dwell < q1 - 1.5 * iqr)
        | (dwell > q3 + 1.5 * iqr)
    )

    print("Dwell-time outliers:", df['Dwell_Outlier'].sum())

Invalid GPS records: 391
Dwell-time outliers: 31


## **Scenario 14 :Build carrier scorecards with consistent denominator and service-level definitions.**

In [67]:
#Compare carriers using shipment volume, delivery performance, and cost.

df['Event_TS_UTC'] = pd.to_datetime(
    df['Event_TS_UTC'], errors='coerce', utc=True
)

df['Promised_Delivery_TS'] = pd.to_datetime(
    df['Promised_Delivery_TS'], errors='coerce', utc=True
)

events = df['Event_Code'].astype('string').str.upper().str.strip()

deliveries = df[
    events.eq('DELIVERED') & df['Event_TS_UTC'].notna()
].copy()

actual_delivery = deliveries.groupby('Shipment_ID')['Event_TS_UTC'].max()

promised_delivery = df.groupby('Shipment_ID')[
    'Promised_Delivery_TS'
].min()

carrier = df.groupby('Shipment_ID').agg(
    Carrier=('Carrier', 'first'),
    Freight_Cost=('Freight_Cost', 'max'),
    Surcharge=('Surcharge', 'max')
)

scorecard = carrier.join(actual_delivery.rename('Actual_Delivery'))
scorecard = scorecard.join(promised_delivery.rename('Promised_Delivery'))

scorecard['On_Time'] = (
    scorecard['Actual_Delivery'].notna()
    & scorecard['Promised_Delivery'].notna()
    & (scorecard['Actual_Delivery'] <= scorecard['Promised_Delivery'])
)

result = scorecard.reset_index().groupby('Carrier').agg(
    Total_Shipments=('Shipment_ID', 'nunique'),
    On_Time_Shipments=('On_Time', 'sum'),
    Total_Freight=('Freight_Cost', 'sum'),
    Total_Surcharge=('Surcharge', 'sum')
).reset_index()

result['On_Time_Rate'] = (
    result['On_Time_Shipments'] / result['Total_Shipments'] * 100
)

display(result)

,Carrier,Total_Shipments,On_Time_Shipments,Total_Freight,Total_Surcharge,On_Time_Rate
0,BLUEJET,437,63,109663.00,7705.0,14.416476
1,FASTSHIP,540,71,132980.05,9325.0,13.148148
2,ODISHAEXPRESS,400,62,99350.97,7355.0,15.500000
3,ROADX,539,89,134354.79,9290.0,16.512059


## **Scenario 15 : Calculate COD collection and reconciliation metrics where relevant.**

In [68]:
#Identify shipments that need cash-on-delivery reconciliation.

df['COD_Amount'] = pd.to_numeric(
    df['COD_Amount'], errors='coerce'
)

cod_summary = df.groupby('Shipment_ID').agg(
    Carrier=('Carrier', 'first'),
    COD_Amount=('COD_Amount', 'max'),
    Currency=('Currency', 'first')
).reset_index()

cod_summary['COD_Issue'] = np.select(
    [
        cod_summary['COD_Amount'].isna(),
        cod_summary['COD_Amount'] < 0
    ],
    [
        'Missing COD Amount',
        'Negative COD Amount'
    ],
    default='Requires Collection Verification'
)

display(cod_summary)

,Shipment_ID,Carrier,COD_Amount,Currency,COD_Issue
0,SHP-10001,FASTSHIP,500.0,INR,Requires Collection Verification
1,SHP-10002,ROADX,0.0,INR,Requires Collection Verification
2,SHP-10003,ROADX,1200.0,INR,Requires Collection Verification
3,SHP-10004,FASTSHIP,0.0,INR,Requires Collection Verification
4,SHP-10005,BLUEJET,0.0,INR,Requires Collection Verification
...,...,...,...,...,...
1911,TEST-11634,BLUEJET,1500.0,INR,Requires Collection Verification
1912,TEST-11708,BLUEJET,0.0,INR,Requires Collection Verification
1913,TEST-11762,ODISHAEXPRESS,2499.0,inr,Requires Collection Verification
1914,TEST-11888,FASTSHIP,0.0,INR,Requires Collection Verification


## **Scenario 16 : Produce privacy-safe operational dashboards without full customer addresses or phone numbers.**

In [69]:
#Create an aggregated dataset without customer names, phone numbers, or addresses.

df['Event_TS_UTC'] = pd.to_datetime(
    df['Event_TS_UTC'], errors='coerce', utc=True
)

df['Report_Date'] = df['Event_TS_UTC'].dt.date

dashboard = df.groupby(
    ['Report_Date', 'Carrier', 'Origin_Depot', 'Destination_Zone']
).agg(
    Total_Shipments=('Shipment_ID', 'nunique'),
    Total_Parcels=('Parcel_Count', 'sum'),
    Average_Weight=('Weight', 'mean'),
    Total_Freight=('Freight_Cost', 'sum')
).reset_index()

personal_columns = [
    'Customer_Name',
    'Customer_Phone',
    'Street_Address',
    'Tracking_ID',
    'Shipment_ID'
]

dashboard.drop(
    columns=personal_columns,
    errors='ignore',
    inplace=True
)

display(dashboard.head(10))

dashboard.to_csv(
    'privacy_safe_dashboard.csv',
    index=False
)

print("Dashboard saved successfully.")

,Report_Date,Carrier,Origin_Depot,Destination_Zone,Total_Shipments,Total_Parcels,Average_Weight,Total_Freight
0,2026-07-01,BLUEJET,NORTH,BBSR-751001,1,1.0,3.66,409.49
1,2026-07-01,BLUEJET,WEST,BBSR-751001,1,1.0,9.45,0.00
2,2026-07-01,FASTSHIP,EAST,RKL-769001,1,1.0,0.77,198.16
3,2026-07-01,ODISHAEXPRESS,EAST,RKL-769001,1,1.0,5.69,404.82
4,2026-07-01,ODISHAEXPRESS,NORTH,BLS-756001,1,1.0,23150.00,0.00
5,2026-07-01,ODISHAEXPRESS,NORTH,UNKNOWN,1,1.0,21580.00,393.82
6,2026-07-01,ROADX,SOUTH,751001-BBSR,1,1.0,16.09,181.10
7,2026-07-01,ROADX,WEST,BAM-760001,1,1.0,8.63,409.08
8,2026-07-02,BLUEJET,NORTH,BAM-760001,1,1.0,10.14,347.00
9,2026-07-02,BLUEJET,NORTH,BLS-756001,1,1.0,9.61,0.00


Dashboard saved successfully.


In [70]:
df

,Shipment_ID,Parent_Shipment_ID,Order_Status,Tracking_ID,Parcel_ID,Parcel_Count,Carrier,Service_Level,Scan_Source,Event_TS,...,Transit,Delivery,Damaged,Lost,Returned,Latitude,Longitude,Invalid_GPS,Dwell_Outlier,Report_Date
4624,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,DRIVER_APP,2026-07-27T07:26:00Z,...,False,False,False,False,False,21.49,83.9631,False,False,NaT
4625,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,API_WEBHOOK,2026-07-27T14:41:00Z,...,False,False,False,False,False,21.4619,83.9559,False,False,NaT
4626,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,HUB_SCANNER,2026-07-27T22:19:00Z,...,False,False,False,False,False,21.4787,84.0084,False,False,NaT
4627,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,BLUEJET,NEXTDAY,MANUAL_ENTRY,2026-07-28T03:40:00Z,...,False,False,False,False,False,<NA>,<NA>,<NA>,False,NaT
4628,SHP-10886,NaN,ACTIVE,TRK-10886,PCL-10886,NaN,ROADX,NEXTDAY,API_WEBHOOK,2026-07-28T06:55:00Z,...,False,True,False,False,False,21.4373,84.0011,False,False,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2777,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,DRIVER_APP,2026-07-25T15:44:00Z,...,False,False,False,False,False,19.8325,85.8324,False,False,NaT
2778,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,API_WEBHOOK,2026-07-26 04:29 +05:30,...,False,False,False,False,False,19.8169,85.8519,False,False,2026-07-25
2779,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,API_WEBHOOK,2026-07-26T03:15:00Z,...,True,False,False,False,False,19.8374,85.8402,False,False,NaT
2780,SHP-10534,NaN,ACTIVE,TRK-10534,PCL-10534,1.0,BLUEJET,REGULAR,MANUAL_ENTRY,2026-07-26T12:33:00Z,...,False,False,False,False,False,19.7927,85.8547,False,False,NaT


## **Checking the Test Cases**

In [10]:
test_results = []

def record_test(test_id, description, condition):
    status = 'PASS' if condition else 'FAIL'

    test_results.append({
        'Test_ID': test_id,
        'Description': description,
        'Result': status
    })

    print(f"{test_id}: {status} - {description}")


# TC01: Tracking ID cleanup
tracking_id = ' trk-9001 '.strip().upper()

record_test(
    'TC01',
    'Tracking ID normalized',
    tracking_id == 'TRK-9001'
)


# TC02: Equivalent weight values
weight_grams = 1500 / 1000
weight_kg = 1.5

record_test(
    'TC02',
    'Equivalent weights match',
    np.isclose(weight_grams, weight_kg)
)


# TC03: Timezone conversion
timestamp = pd.to_datetime(
    '2026-07-12T09:00:00+05:30',
    utc=True
)

record_test(
    'TC03',
    'Timezone converted to 03:30 UTC',
    timestamp.hour == 3 and timestamp.minute == 30
)


# TC04: Missing timezone
timestamp_text = '2026-07-13 08:00:00'

has_timezone = bool(
    pd.Series([timestamp_text])
    .str.contains(r'(?:Z|[+-]\d{2}:?\d{2})$', case=False)
    .iloc[0]
)

record_test(
    'TC04',
    'Missing timezone detected',
    not has_timezone
)


# TC05: Negative weight
weight = -4

record_test(
    'TC05',
    'Negative weight rejected',
    weight < 0
)


# TC06: Invalid GPS
latitude = 125
longitude = -220

gps_invalid = (
    not -90 <= latitude <= 90
    or not -180 <= longitude <= 180
)

record_test(
    'TC06',
    'Invalid GPS detected',
    gps_invalid
)


# TC07: Event ordering
pickup_time = pd.Timestamp('2026-07-12 10:00:00', tz='UTC')
delivery_time = pd.Timestamp('2026-07-12 09:00:00', tz='UTC')

record_test(
    'TC07',
    'Delivery before pickup detected',
    delivery_time < pickup_time
)


# TC08: Carrier mapping
carrier = 'FAST SHIP'.strip().upper().replace(' ', '')

record_test(
    'TC08',
    'Carrier mapped to FASTSHIP',
    carrier == 'FASTSHIP'
)


# TC09: Duplicate active tracking ID
active_tracking_ids = pd.Series(['TRK-100', 'TRK-100'])

record_test(
    'TC09',
    'Duplicate tracking ID detected',
    active_tracking_ids.duplicated().any()
)


# TC10: Late delivery
promised = pd.Timestamp('2026-07-12 10:00:00', tz='UTC')
actual = pd.Timestamp('2026-07-12 11:00:00', tz='UTC')

record_test(
    'TC10',
    'Late delivery detected',
    actual > promised
)


# Display final results
test_report = pd.DataFrame(test_results)

display(test_report)

print("\nTest summary:")
print(test_report['Result'].value_counts())

test_report.to_csv(
    'test_case_results.csv',
    index=False
)

TC01: PASS - Tracking ID normalized
TC02: PASS - Equivalent weights match
TC03: PASS - Timezone converted to 03:30 UTC
TC04: PASS - Missing timezone detected
TC05: PASS - Negative weight rejected
TC06: PASS - Invalid GPS detected
TC07: PASS - Delivery before pickup detected
TC08: PASS - Carrier mapped to FASTSHIP
TC09: PASS - Duplicate tracking ID detected
TC10: PASS - Late delivery detected


,Test_ID,Description,Result
0,TC01,Tracking ID normalized,PASS
1,TC02,Equivalent weights match,PASS
2,TC03,Timezone converted to 03:30 UTC,PASS
3,TC04,Missing timezone detected,PASS
4,TC05,Negative weight rejected,PASS
5,TC06,Invalid GPS detected,PASS
6,TC07,Delivery before pickup detected,PASS
7,TC08,Carrier mapped to FASTSHIP,PASS
8,TC09,Duplicate tracking ID detected,PASS
9,TC10,Late delivery detected,PASS



Test summary:
Result
PASS    10
Name: count, dtype: int64
